# Barebone Halofit $P_{mm}$ vs CLASS

**Part 1** — identical to `linear_pk_vs_class`: loads `Pk_lin_mm` and `Pk_lin_cb` components and compares linear spectra against CLASS.

**Part 2** — Halofit comparison with two panels:

- **Panel 1**: CLASS linear $P_{mm}$ + jaxmapse Halofit vs CLASS native Halofit. Isolates the Halofit kernel difference.
- **Panel 2**: Emulated linear $P_{mm}$ + jaxmapse Halofit vs CLASS native Halofit. Full Mapse pipeline.

In [ ]:
import os, sys
from contextlib import contextmanager

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from classy import Class

import jaxmapse

In [ ]:
#--- Cosmology (flat ΛCDM) ---
cosmo = {
    "ln10As":  3.044,
    "ns":      0.9649,
    "H0":      67.36,
    "omega_b": 0.02237,
    "omega_c": 0.12,
    "Mnu":     0.06,
    "w0":     -1.0,
    "wa":      0.0,
}

# Emulator parameter vector order:
# [ln10As, ns, H0, omega_b, omega_c, Mnu, w0, wa]
params = jnp.array([
    cosmo["ln10As"], cosmo["ns"], cosmo["H0"],
    cosmo["omega_b"], cosmo["omega_c"], cosmo["Mnu"],
    cosmo["w0"], cosmo["wa"],
])

z_eval = jnp.array([0.0, 1.0, 2.0, 3.0, 4.0])

In [ ]:
trained = jaxmapse.load_trained_emulators()[jaxmapse.DEFAULT_EMULATOR_ARTIFACT]
pmm = trained["pmm"]
pcb = trained["pcb"]

k = np.asarray(pmm.k_grid)
print("k grid:", k.shape, k.min(), k.max())


In [ ]:
h = cosmo["H0"] / 100.0
jax_cosmo = jaxmapse.w0waCDMCosmology(
    ln10As=cosmo["ln10As"],
    ns=cosmo["ns"],
    h=h,
    omega_b=cosmo["omega_b"],
    omega_c=cosmo["omega_c"],
    m_nu=cosmo["Mnu"],
    w0=cosmo["w0"],
    wa=cosmo["wa"],
)
D_eval = jnp.asarray(jax_cosmo.D_z(z_eval))
print("D(z):", np.asarray(D_eval))

In [ ]:
# Shape: (len(z), len(k))
Pmm_emu = np.asarray(pmm(params, z_eval, D_eval))
Pcb_emu = np.asarray(pcb(params, z_eval, D_eval))
print("Pmm_emu:", Pmm_emu.shape, "  Pcb_emu:", Pcb_emu.shape)

## CLASS reference

In [ ]:
@contextmanager
def suppress_c_output():
    sys.stdout.flush(); sys.stderr.flush()
    old1, old2 = os.dup(1), os.dup(2)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), 1); os.dup2(devnull.fileno(), 2)
            yield
    finally:
        os.dup2(old1, 1); os.dup2(old2, 2)
        os.close(old1); os.close(old2)

def class_params(cosmo, z_max, k_max, nonlinear=True):
    p = {
        "output": "mPk",
        "P_k_max_1/Mpc": float(k_max),
        "z_max_pk": float(z_max),
        "h": cosmo["H0"] / 100.0,
        "omega_b": cosmo["omega_b"],
        "omega_cdm": cosmo["omega_c"],
        "N_ur": 2.0328,
        "N_ncdm": 1,
        "m_ncdm": cosmo["Mnu"],
        "tau_reio": 0.0544,
        "A_s": float(np.exp(cosmo["ln10As"]) * 1e-10),
        "n_s": cosmo["ns"],
        "w0_fld": cosmo["w0"],
        "wa_fld": cosmo["wa"],
        "Omega_Lambda": 0.0,
        "fluid_equation_of_state": "CLP",
        "use_ppf": "yes",
    }
    if nonlinear:
        p["non linear"] = "halofit"
    return p

def run_class(cosmo, z_eval, k, nonlinear=True):
    c = Class()
    c.set(class_params(cosmo, z_max=float(np.max(z_eval)) + 0.5,
                        k_max=float(np.max(k) * 1.05), nonlinear=nonlinear))
    with suppress_c_output():
        c.compute()
        Pmm_lin = np.array([[c.pk_lin(float(kk), float(zz)) for kk in k] for zz in z_eval])
        Pcb_lin = np.array([[c.pk_cb_lin(float(kk), float(zz)) for kk in k] for zz in z_eval])
        if nonlinear:
            Pmm_nl = np.array([[c.pk(float(kk), float(zz)) for kk in k] for zz in z_eval])
        else:
            Pmm_nl = None
    c.struct_cleanup(); c.empty()
    return Pmm_lin, Pcb_lin, Pmm_nl

In [ ]:
# Run CLASS with Halofit enabled so we get linear + nonlinear in one shot
Pmm_class, Pcb_class, Pmm_class_nl = run_class(cosmo, np.asarray(z_eval), k, nonlinear=True)
print("CLASS Pmm lin:", Pmm_class.shape, "  CLASS Pcb lin:", Pcb_class.shape)
print("CLASS Pmm nl:", Pmm_class_nl.shape)

## Linear residuals

In [ ]:
mask = (k >= 1e-4) & (k <= 10.0)
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(z_eval)))

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex="col")

for col, (title, emu_pk, class_pk) in enumerate([
    (r"Linear $P_{mm}$", Pmm_emu, Pmm_class),
    (r"Linear $P_{cb}$", Pcb_emu, Pcb_class),
]):
    ax_top = axes[0, col]
    ax_res = axes[1, col]
    for iz, (zz, color) in enumerate(zip(z_eval, colors)):
        ax_top.loglog(k[mask], class_pk[iz, mask], color=color, lw=2, label=f"CLASS z={zz:g}")
        ax_top.loglog(k[mask],   emu_pk[iz, mask], color=color, ls="--", lw=1.5, label=f"emu z={zz:g}")
        rel = 100.0 * (emu_pk / class_pk - 1.0)
        ax_res.semilogx(k[mask], rel[iz, mask], color=color, lw=1.5)
    ax_top.set_title(title)
    ax_top.set_ylabel(r"$P(k)\;[{\rm Mpc}^3]$")
    ax_top.grid(True, which="both", alpha=0.25)
    ax_res.axhline(0.0, color="k", lw=0.8)
    ax_res.set_xlabel(r"$k\,[{\rm Mpc}^{-1}]$")
    ax_res.set_ylabel("residual [%]")
    ax_res.grid(True, which="both", alpha=0.25)

axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## Halofit comparison

- **Panel 1**: CLASS linear $P_{mm}$ fed into jaxmapse Halofit, compared to CLASS native nonlinear $P_{mm}$. Isolates Halofit kernel differences.
- **Panel 2**: Emulated linear $P_{mm}$ fed into jaxmapse Halofit, compared to CLASS native nonlinear $P_{mm}$. Full Mapse pipeline.

In [ ]:
# jaxmapse Halofit background for our cosmology
halofit_cosmo = jaxmapse.halofit_cosmology(params)
omega_m_z, omega_v_z = jaxmapse.halofit_background(halofit_cosmo, z_eval)
omega_m_z = np.asarray(omega_m_z)
omega_v_z = np.asarray(omega_v_z)

# Panel 1: CLASS linear Pmm + jaxmapse Halofit
Pmm_halofit_class_lin = np.asarray(jaxmapse.halofit_pmm(
    halofit_cosmo, z_eval, jnp.asarray(k), jnp.asarray(Pmm_class),
    jnp.asarray(omega_m_z), jnp.asarray(omega_v_z),
))

# Panel 2: Emulated linear Pmm + jaxmapse Halofit
Pmm_halofit_emu_lin = np.asarray(jaxmapse.halofit_pmm(
    halofit_cosmo, z_eval, jnp.asarray(k), jnp.asarray(Pmm_emu),
    jnp.asarray(omega_m_z), jnp.asarray(omega_v_z),
))

print("Halofit (CLASS lin):", Pmm_halofit_class_lin.shape)
print("Halofit (emu  lin):", Pmm_halofit_emu_lin.shape)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex="col")

panels = [
    (r"CLASS $P_{mm}^{\rm lin}$ + jaxmapse Halofit\nvs CLASS native Halofit",
     Pmm_halofit_class_lin, Pmm_class_nl),
    (r"Emulated $P_{mm}^{\rm lin}$ + jaxmapse Halofit\nvs CLASS native Halofit",
     Pmm_halofit_emu_lin, Pmm_class_nl),
]

for col, (title, model, reference) in enumerate(panels):
    ax_top = axes[0, col]
    ax_res = axes[1, col]
    for iz, (zz, color) in enumerate(zip(z_eval, colors)):
        ax_top.loglog(k[mask], reference[iz, mask], color=color, lw=2, label=f"CLASS Halofit z={zz:g}")
        ax_top.loglog(k[mask],    model[iz, mask], color=color, ls="--", lw=1.5, label=f"jaxmapse z={zz:g}")
        rel = 100.0 * (model / reference - 1.0)
        ax_res.semilogx(k[mask], rel[iz, mask], color=color, lw=1.5)
    ax_top.set_title(title, fontsize=11)
    ax_top.set_ylabel(r"$P_{mm}^{\rm nl}(k)\;[{\rm Mpc}^3]$")
    ax_top.grid(True, which="both", alpha=0.25)
    ax_res.axhline(0.0, color="k", lw=0.8)
    ax_res.set_xlabel(r"$k\,[{\rm Mpc}^{-1}]$")
    ax_res.set_ylabel("residual [%]")
    ax_res.grid(True, which="both", alpha=0.25)

axes[0, 0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## Vectorized Halofit timing

This section uses 100 redshifts between 0.0 and 3.5 to test the batched Halofit path and benchmark it with `%timeit`.

In [ ]:
z_bench = jnp.linspace(0.0, 3.5, 200)
D_bench = jax_cosmo.D_z(z_bench)

# Warm up / compile the vectorized path.
k_vec, pk_vec = jaxmapse.halofit_pmm_from_emulator(params, z_bench, D_bench, linear_pmm_emu=pmm)
jax.block_until_ready((k_vec, pk_vec))
print("vectorized z shape:", z_bench.shape, "pk shape:", pk_vec.shape)

# Time the batched 100-redshift prediction path.
%timeit -n 50 -r 20 jax.block_until_ready(jaxmapse.halofit_pmm_from_emulator(params, z_bench, D_bench, linear_pmm_emu=pmm))

## JIT-compiled vectorized Halofit

The batched growth factor and batched Halofit call are both JAX-compatible. The cell below JIT-compiles the full 100-redshift prediction path.

In [ ]:
z_jit = jnp.linspace(0.0, 3.5, 200)

@jax.jit
def jitted_vectorized_halofit(params, z):
    # One JAX call for the full redshift array. The ODE/growth solve is
    # handled once for the whole batch, then fed into the emulator + Halofit.
    D = jax_cosmo.D_z(z)
    return jaxmapse.halofit_pmm_from_emulator(params, z, D, linear_pmm_emu=pmm)

# Compile and sanity-check.
k_jit, pk_jit = jitted_vectorized_halofit(params, z_jit)
jax.block_until_ready((k_jit, pk_jit))
print("JIT vectorized z shape:", z_jit.shape, "pk shape:", pk_jit.shape)

# Benchmark the JIT-compiled vectorized path.
%timeit -n 50 -r 20 jax.block_until_ready(jitted_vectorized_halofit(params, z_jit))